### 사전학습 임베딩 사용, 전이학습 고려
* DistilBertModel 사용

In [ ]:
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

from transformers import DistilBertTokenizerFast, DistilBertModel

DATA_PATH = "dataset/emotion_recognitions_merged.csv"
TEXT_COL = "text"
LABEL_COL = "label"

EPOCHS = 10
BATCH_SIZE = 64
LR = 2e-5
MAX_LEN = 128     # DistilBERT 권장 128~256
NUM_CLASSES = 6
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Dataset 준비 (Tokenizer 사용)
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

class DistilBertEmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.encodings = tokenizer(texts,
                                   truncation=True,
                                   padding='max_length',
                                   max_length=max_len,
                                   return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

# 모델 정의 (DistilBERT + Classification head)
class DistilBERTClassifier(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, dropout_rate=0.2):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        hidden_size = self.distilbert.config.hidden_size  # 보통 768
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(hidden_size, num_classes)

        nn.init.xavier_uniform_(self.classifier.weight)

    def forward(self, input_ids, attention_mask):
        # DistilBERT outputs: last_hidden_state (batch, seq_len, hidden)
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        # pooled: 일반적으로 DistilBERT는 CLS token embedding을 사용 (token 0)
        # last_hidden_state[:,0,:] 을 사용하여 문장 표현으로 활용
        last_hidden = outputs.last_hidden_state  # shape: (batch, seq_len, hidden)
        pooled = last_hidden[:, 0, :]           # CLS 토큰 위치 (batch, hidden)
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)         # (batch, num_classes)
        return logits

# 학습, 평가 함수 (평가 지표는 동일)
def train_model(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * input_ids.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids']
            attention_mask = batch['attention_mask']
            labels = batch['labels']

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            pred = torch.argmax(outputs, dim=1)

            preds.extend(pred.numpy())
            trues.extend(labels.numpy())

    print(classification_report(trues, preds, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(trues, preds))

if __name__ == "__main__":
    df = pd.read_csv(DATA_PATH)
    texts = df[TEXT_COL].astype(str).tolist()
    labels = df[LABEL_COL].astype(int).tolist()

    # train, val, test split
    X_train, X_temp, y_train, y_temp = train_test_split(
        texts, labels, test_size=0.3, stratify=labels, random_state=SEED
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED
    )

    # Dataset, DataLoader (토크나이저로 미리 인코딩)
    train_ds = DistilBertEmotionDataset(X_train, y_train, tokenizer, max_len=MAX_LEN)
    val_ds = DistilBertEmotionDataset(X_val, y_val, tokenizer, max_len=MAX_LEN)
    test_ds = DistilBertEmotionDataset(X_test, y_test, tokenizer, max_len=MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

    # 모델, 손실함수 (클래스 가중치 적용), 옵티마이저
    model = DistilBERTClassifier(num_classes=NUM_CLASSES)

    # 클래스 가중치 계산 (train 기준)
    classes = np.unique(y_train)
    class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    weight_tensor = torch.zeros(NUM_CLASSES, dtype=torch.float)
    for i, c in enumerate(classes):
        weight_tensor[c] = class_weights[i]

    criterion = nn.CrossEntropyLoss(weight=weight_tensor)
    optimizer = optim.AdamW(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        train_loss = train_model(model, train_loader, criterion, optimizer)
        print(f"\nEpoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f}")
        print(f"Epoch {epoch+1} Validation:")
        evaluate(model, val_loader)

    print("\nFinal Test Performance:")
    evaluate(model, test_loader)

Epoch 1/10 - Train Loss: 0.1706
Epoch 1 Validation:
              precision    recall  f1-score   support

           0     0.9984    0.9562    0.9768     19047
           1     0.9974    0.9147    0.9542     22174
           2     0.7695    0.9993    0.8695      5429
           3     0.9290    0.9653    0.9468      9004
           4     0.9142    0.8891    0.9015      7513
           5     0.7274    0.9987    0.8417      2354

    accuracy                         0.9408     65521
   macro avg     0.8893    0.9539    0.9151     65521
weighted avg     0.9502    0.9408    0.9427     65521

Confusion Matrix:
[[18212    22     1   464   333    15]
 [   13 20282  1624    41     4   210]
 [    0     3  5425     1     0     0]
 [    8    14     0  8692   290     0]
 [    7    12     0   158  6680   656]
 [    1     2     0     0     0  2351]]